### 1. Carga de Librerías y Datos

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import gdown
import os

# Carga de los Datos desde Google Drive usando gdown para descargarlo localmente
file_id = '119DwKkes8OjM6pBuQ2gDGuYpYYegXG7u'
url = f'https://drive.google.com/uc?id={file_id}'
output_file = 'dataset_tratamiento.xlsx'

if not os.path.exists(output_file):
    gdown.download(url, output_file, quiet=False)

df = pd.read_excel(output_file)

# Renombrar columnas para usar "alias" y coincidir exactamente con la app
df = df.rename(columns={
    '7in1_Nitrogen[mg/kg]': 'nitrogeno',
    '7in1_Phosphorus[mg/kg]': 'fosforo',
    '7in1_Potasium[mg/kg]': 'potasio',
    '7in1_Ph[pH]': 'ph',
    '7in1_Conductivity[uS/cm]': 'conductividad',
    '7in1_S_Temperature[C]': 'temperatura_suelo',
    '7in1_Moisture[%RH]': 'humedad_suelo',
    'Air_sensor_Temperature[C]': 'temperatura',
    'Air_sensor_Humidity[%RH]': 'humedad'
})

# Definición de variables con los nuevos nombres
col_n = 'nitrogeno'
col_p = 'fosforo'
col_k = 'potasio'
col_ph = 'ph'
col_ec = 'conductividad'
col_temp_suelo = 'temperatura_suelo'
col_hum_suelo = 'humedad_suelo'
col_temp = 'temperatura'
col_hum = 'humedad'
col_tratamiento = 'Tratamiento'

display(df.head())


Downloading...
From: https://drive.google.com/uc?id=119DwKkes8OjM6pBuQ2gDGuYpYYegXG7u
To: /content/dataset_tratamiento.xlsx
100%|██████████| 1.15M/1.15M [00:00<00:00, 25.2MB/s]


,Fecha,Laboratorio,Cultivo,Planta,Time,nitrogeno,fosforo,potasio,ph,humedad_suelo,...,Pynamometer_Radiation[W/m^2],temperatura,humedad,RGB_capture,Tapo_capture,Parrot_capture,Posicion_x,Posicion_y,Tratamiento,Tratamiento iniciado
0,2024-09-11,Quindio,Tomatoes,121,2024-09-11 11:50:00,522.0,644.0,1214.0,6.30,36.9,...,24.6,29.0,52.5,1,1,5,1.0,12.0,T5_N0P0K2,Si
1,2024-09-11,Quindio,Tomatoes,145,2024-09-11 11:53:00,272.0,373.0,693.0,9.00,29.6,...,28.2,28.7,52.9,1,1,5,1.0,14.4,T6_N0P1K1,Si
2,2024-09-11,Quindio,Tomatoes,165,2024-09-11 11:58:00,161.0,228.0,451.0,6.33,20.4,...,26.1,28.5,56.5,1,1,5,3.2,15.6,T8_N0P2K1,Si
3,2024-09-11,Quindio,Tomatoes,172,2024-09-11 12:24:00,140.0,198.0,397.0,6.40,23.0,...,30.4,29.7,52.6,1,1,5,3.2,16.8,T11_N1P1K0,Si
4,2024-08-14,Quindio,Tomatoes,73,2024-08-14 10:24:00,95.0,140.0,288.0,6.50,26.1,...,28.9,25.8,66.3,1,1,5,1.0,7.2,T19_N2P0K0,Si


### 2. Configuración del Tratamiento a Analizar

In [2]:
# Define aquí el/los tratamiento(s) que deseas analizar.
# - Un solo tratamiento: 'N1P1K1'
# - Múltiples tratamientos separados por comas: 'N1P1K1, N0P0K0, N2P2K2'
# - Todos los datos sin filtrar: 'todos'

tratamiento_seleccionado = 'todos'


### 3. Procesamiento de Datos, PCA y Entrenamiento de Modelos

In [3]:
# Filtro del dataset basado en la selección manual
seleccion = str(tratamiento_seleccionado).strip().lower()

if seleccion == 'todos':
    df_filtered = df.copy()
    nombres_tratamientos = "Todos"
else:
    # Separar por comas y limpiar espacios vacíos
    lista_tratamientos = [t.strip() for t in str(tratamiento_seleccionado).split(',')]

    # Crear máscara dinámica iterando sobre los tratamientos
    mask = pd.Series(False, index=df.index)
    for t in lista_tratamientos:
        mask = mask | (df[col_tratamiento] == t) | (df[col_tratamiento].str.contains(t, na=False))

    df_filtered = df[mask].copy()
    nombres_tratamientos = ", ".join(lista_tratamientos)

print(f"✅ Se encontraron {len(df_filtered)} registros para la selección '{nombres_tratamientos}'.")

if len(df_filtered) < 5:
    raise ValueError("❌ Muy pocos datos para procesar el modelo con esta selección.")
elif len(df_filtered) < 50:
    print("⚠️ Advertencia: Tienes pocos registros para esta selección. Los modelos requieren al menos 50 para ser estables.")

# --- NUEVO: Generación de variables temporales cíclicas (senos y cosenos) en minúsculas ---
df_filtered['Time'] = pd.to_datetime(df_filtered['Time'])
df_filtered['hora_sin'] = np.sin(2 * np.pi * df_filtered['Time'].dt.hour / 24.0)
df_filtered['hora_cos'] = np.cos(2 * np.pi * df_filtered['Time'].dt.hour / 24.0)
df_filtered['mes_sin'] = np.sin(2 * np.pi * df_filtered['Time'].dt.month / 12.0)
df_filtered['mes_cos'] = np.cos(2 * np.pi * df_filtered['Time'].dt.month / 12.0)

# Limpieza de nulos en variables clave incluyendo las nuevas variables temporales
features = [col_n, col_p, col_k, col_ph, col_ec, col_hum_suelo, col_temp_suelo, col_temp, col_hum, 'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos']
df_clean = df_filtered.dropna(subset=features).copy()

# 4. Agrupación por Dominios y Reducción PCA
grupo_nutrientes = [col_n, col_p, col_k, col_ph, col_ec]
grupo_suelo = [col_hum_suelo, col_temp_suelo]
# Incluimos las variables de tiempo en el grupo de ambiente
grupo_ambiente = [col_temp, col_hum, 'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos']

# Instanciar y guardar los scalers (útiles para predecir datos futuros)
scaler_nut = StandardScaler()
scaler_suelo = StandardScaler()
scaler_amb = StandardScaler()

scaled_nut = scaler_nut.fit_transform(df_clean[grupo_nutrientes])
scaled_suelo = scaler_suelo.fit_transform(df_clean[grupo_suelo])
scaled_amb = scaler_amb.fit_transform(df_clean[grupo_ambiente])

# Instanciar y guardar los PCA
pca_nut = PCA(n_components=1, random_state=42)
pca_suelo = PCA(n_components=1, random_state=42)
pca_amb = PCA(n_components=1, random_state=42)

comp_nut = pca_nut.fit_transform(scaled_nut)
comp_suelo = pca_suelo.fit_transform(scaled_suelo)
comp_amb = pca_amb.fit_transform(scaled_amb)

X_pca = pd.DataFrame({
    'Comp_Nutrientes': comp_nut.ravel(),
    'Comp_Suelo': comp_suelo.ravel(),
    'Comp_Ambiente': comp_amb.ravel()
}, index=df_clean.index)

df_clean[['Comp_Nutrientes', 'Comp_Suelo', 'Comp_Ambiente']] = X_pca

# 5. Entrenamiento de Isolation Forest (Usa los componentes PCA)
iforest = IsolationForest(n_estimators=500, contamination=0.1, random_state=42)
df_clean['Pred_IF'] = iforest.fit_predict(X_pca)

# --- Lógica de asignación basada en Reglas Óptimas (Tomate) ---
RANGES_SANA = {
    "temperatura": (18.0, 30.0),
    "humedad": (44.0, 75.0),
    "nitrogeno": (10, 50),
    "fosforo": (15, 60),
    "potasio": (40, 250),
    "humedad_suelo": (15.0, 45.0),
    "temperatura_suelo": (20.0, 30.0),
    "conductividad": (100, 500),
    "ph": (5.8, 6.8),
}

def diagnosticar_por_reglas(row):
    # Ambiente
    if not (RANGES_SANA["temperatura"][0] <= row[col_temp] <= RANGES_SANA["temperatura"][1]) or \
       not (RANGES_SANA["humedad"][0] <= row[col_hum] <= RANGES_SANA["humedad"][1]):
        return 'Anomalía de Ambiente'
    # Suelo
    if not (RANGES_SANA["humedad_suelo"][0] <= row[col_hum_suelo] <= RANGES_SANA["humedad_suelo"][1]) or \
       not (RANGES_SANA["temperatura_suelo"][0] <= row[col_temp_suelo] <= RANGES_SANA["temperatura_suelo"][1]):
        return 'Anomalía de Suelo'
    # Nutrientes
    if not (RANGES_SANA["nitrogeno"][0] <= row[col_n] <= RANGES_SANA["nitrogeno"][1]) or \
       not (RANGES_SANA["fosforo"][0] <= row[col_p] <= RANGES_SANA["fosforo"][1]) or \
       not (RANGES_SANA["potasio"][0] <= row[col_k] <= RANGES_SANA["potasio"][1]) or \
       not (RANGES_SANA["conductividad"][0] <= row[col_ec] <= RANGES_SANA["conductividad"][1]) or \
       not (RANGES_SANA["ph"][0] <= row[col_ph] <= RANGES_SANA["ph"][1]):
        return 'Anomalía de Nutrientes'

    return 'Sin anomalías (Sana)'

# Mantenemos el nombre de la columna original para no romper los gráficos siguientes
df_clean['Diagnostico_IF'] = df_clean.apply(diagnosticar_por_reglas, axis=1)

# 6. Entrenamiento de Random Forest con las 3 componentes PCA (en lugar de las 13 variables originales)
features_pca = ['Comp_Nutrientes', 'Comp_Suelo', 'Comp_Ambiente']
rf = RandomForestClassifier(n_estimators=500, random_state=42, oob_score=True)
rf.fit(df_clean[features_pca], df_clean['Diagnostico_IF'])
df_clean['Pred_RF'] = rf.predict(df_clean[features_pca])

# Definir mapa de colores para las visualizaciones
color_map = {
    'Sin anomalías (Sana)': '#2E7D32',
    'Anomalía de Nutrientes': '#C62828',
    'Anomalía de Suelo': '#F57F17',
    'Anomalía de Ambiente': '#1565C0'
}

✅ Se encontraron 1025 registros para la selección 'Todos'.


### 4. Visualización 3D Comparativa de Anomalías

In [4]:

# 1. Gráfico de asignación de anomalías por Isolation Forest
fig_if = px.scatter_3d(
    df_clean,
    x='Comp_Nutrientes',
    y='Comp_Suelo',
    z='Comp_Ambiente',
    color='Diagnostico_IF',
    color_discrete_map={
        'Sin anomalías (Sana)': '#2E7D32',
        'Anomalía de Nutrientes': '#C62828',
        'Anomalía de Suelo': '#F57F17',
        'Anomalía de Ambiente': '#1565C0'
    },
    title=f"1. Asignación de Anomalías por Isolation Forest ({tratamiento_seleccionado})",
    hover_data=['Time'] + features,
    labels={
        'Comp_Nutrientes': 'Componente Nutrientes',
        'Comp_Suelo': 'Componente Suelo',
        'Comp_Ambiente': 'Componente Ambiente'
    }
)
fig_if.update_traces(marker=dict(size=6, opacity=0.8))
fig_if.show()

# 2. Gráfico de clasificación aprendida por Random Forest
fig_rf = px.scatter_3d(
    df_clean,
    x='Comp_Nutrientes',
    y='Comp_Suelo',
    z='Comp_Ambiente',
    color='Pred_RF',
    color_discrete_map={
        'Sin anomalías (Sana)': '#2E7D32',
        'Anomalía de Nutrientes': '#C62828',
        'Anomalía de Suelo': '#F57F17',
        'Anomalía de Ambiente': '#1565C0'
    },
    title=f"2. Clasificación Aprendida por Random Forest ({tratamiento_seleccionado})",
    hover_data=['Time'] + features,
    labels={
        'Comp_Nutrientes': 'Nutrientes',
        'Comp_Suelo': 'Suelo',
        'Comp_Ambiente': 'Ambiente'
    }
)

fig_rf.update_traces(marker=dict(size=6, opacity=0.8))
fig_rf.show()

### 5. Evaluación de Precisión del Modelo

Analizaremos la precisión de la clasificación del modelo **Random Forest** entrenado en base a los diagnósticos derivados de las anomalías detectadas por **Isolation Forest**.

El Random Forest recibe únicamente las 3 componentes PCA. El reporte se calcula sobre los mismos datos de entrenamiento (un Random Forest tiende a acertar ~100% sobre ellos), por lo que también se muestra la exactitud OOB (*out-of-bag*), que es una estimación sobre datos no vistos por cada árbol.

In [5]:
from sklearn.metrics import classification_report, confusion_matrix
import plotly.figure_factory as ff

# 1. Reporte de Clasificación
y_true = df_clean['Diagnostico_IF']
y_pred = df_clean['Pred_RF']

report = classification_report(y_true, y_pred, output_dict=False)
print(f"Exactitud OOB (estimación sobre datos no vistos): {rf.oob_score_ * 100:.2f}%\n")
print("=== Reporte de Clasificación (Random Forest vs. Diagnósticos de Isolation Forest) ===\n")
print(report)

# 2. Matriz de Confusión Normalizada
labels = sorted(list(y_true.unique()))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Generar gráfico interactivo de la matriz de confusión
fig_cm = ff.create_annotated_heatmap(
    z=np.round(cm_normalized, 2),
    x=labels,
    y=labels,
    colorscale='Blues',
    showscale=True
)

fig_cm.update_layout(
    title="Matriz de Confusión Normalizada (Precisión del Modelo)",
    xaxis_title="Predicción (Random Forest)",
    yaxis_title="Real (Isolation Forest)",
    width=600,
    height=500
)

fig_cm.show()

Exactitud OOB (estimación sobre datos no vistos): 81.94%

=== Reporte de Clasificación (Random Forest vs. Diagnósticos de Isolation Forest) ===

                        precision    recall  f1-score   support

  Anomalía de Ambiente       1.00      1.00      1.00       131
Anomalía de Nutrientes       1.00      1.00      1.00       239
     Anomalía de Suelo       1.00      1.00      1.00        42
  Sin anomalías (Sana)       1.00      1.00      1.00       114

              accuracy                           1.00       526
             macro avg       1.00      1.00      1.00       526
          weighted avg       1.00      1.00      1.00       526



### 6. Prueba Interactiva de Predicción

Ingresa valores personalizados para verificar cómo reaccionan los modelos ante diferentes condiciones.

In [6]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import numpy as np

# Crear widgets de entrada para cada variable
w_temp = widgets.FloatText(value=25.0, description='Temp Aire (°C):', style={'description_width': 'initial'})
w_hum = widgets.FloatText(value=60.0, description='Humedad Aire (%):', style={'description_width': 'initial'})
w_n = widgets.FloatText(value=30.0, description='Nitrógeno (mg/kg):', style={'description_width': 'initial'})
w_p = widgets.FloatText(value=30.0, description='Fósforo (mg/kg):', style={'description_width': 'initial'})
w_k = widgets.FloatText(value=100.0, description='Potasio (mg/kg):', style={'description_width': 'initial'})
w_hum_suelo = widgets.FloatText(value=30.0, description='Humedad Suelo (%):', style={'description_width': 'initial'})
w_temp_suelo = widgets.FloatText(value=22.0, description='Temp Suelo (°C):', style={'description_width': 'initial'})
w_ec = widgets.FloatText(value=200.0, description='Conductividad (uS/cm):', style={'description_width': 'initial'})
w_ph = widgets.FloatText(value=6.0, description='pH:', style={'description_width': 'initial'})

btn_predict = widgets.Button(description="Predecir", button_style='success')
output = widgets.Output()

def on_predict_clicked(b):
    with output:
        output.clear_output()
        # Crear DataFrame con los valores de entrada, incluyendo variables de tiempo estáticas de ejemplo
        input_data = pd.DataFrame([{
            col_n: w_n.value,
            col_p: w_p.value,
            col_k: w_k.value,
            col_ph: w_ph.value,
            col_ec: w_ec.value,
            col_hum_suelo: w_hum_suelo.value,
            col_temp_suelo: w_temp_suelo.value,
            col_temp: w_temp.value,
            col_hum: w_hum.value,
            'hora_sin': 0.0, 'hora_cos': 1.0, 'mes_sin': 0.0, 'mes_cos': 1.0
        }])

        # Escalar los nuevos datos (Isolation Forest y Random Forest)
        scaled_nut_in = scaler_nut.transform(input_data[grupo_nutrientes])
        scaled_suelo_in = scaler_suelo.transform(input_data[grupo_suelo])
        scaled_amb_in = scaler_amb.transform(input_data[grupo_ambiente])

        # Reducir con PCA (entrada de ambos modelos)
        comp_nut_in = pca_nut.transform(scaled_nut_in)
        comp_suelo_in = pca_suelo.transform(scaled_suelo_in)
        comp_amb_in = pca_amb.transform(scaled_amb_in)

        X_pca_in = pd.DataFrame({
            'Comp_Nutrientes': comp_nut_in.ravel(),
            'Comp_Suelo': comp_suelo_in.ravel(),
            'Comp_Ambiente': comp_amb_in.ravel()
        })

        # Predecir: ambos modelos usan las 3 componentes PCA
        pred_if = iforest.predict(X_pca_in)[0]
        pred_rf = rf.predict(X_pca_in[features_pca])[0]

        if_label = "Sin anomalías (1)" if pred_if == 1 else "Anomalía detectada (-1)"

        print("=== Resultados de la Predicción ===")
        print(f"Isolation Forest (Detección de outliers): {if_label}")
        print(f"Random Forest (Clasificación por reglas): {pred_rf}")

btn_predict.on_click(on_predict_clicked)

form = widgets.VBox([
    widgets.HTML("<b>Ingresa los valores controlados para probar el modelo:</b>"),
    w_temp, w_hum, w_n, w_p, w_k, w_hum_suelo, w_temp_suelo, w_ec, w_ph,
    btn_predict,
    output
])

display(form)

### 7. Exportación de Modelos y Componentes (Formato Joblib)

Exportamos los modelos (`Isolation Forest` y `Random Forest`) junto con los `Scalers` y objetos `PCA` utilizando la librería `joblib` para una mayor eficiencia en la persistencia de datos.

In [7]:
import joblib
from google.colab import files

# 1. Exportar Isolation Forest junto con los preprocesadores requeridos para su Pipeline
iforest_pack = {
    'scaler_nut': scaler_nut,
    'scaler_suelo': scaler_suelo,
    'scaler_amb': scaler_amb,
    'pca_nut': pca_nut,
    'pca_suelo': pca_suelo,
    'pca_amb': pca_amb,
    'iforest': iforest,
    'features': features,
    'grupo_nutrientes': grupo_nutrientes,
    'grupo_suelo': grupo_suelo,
    'grupo_ambiente': grupo_ambiente
}

nombre_iforest = 'modelo_isolation_forest.joblib'
joblib.dump(iforest_pack, nombre_iforest)
print(f"✅ Isolation Forest exportado correctamente en '{nombre_iforest}'.")

# 2. Exportar Random Forest junto con los preprocesadores requeridos (entrada: 3 componentes PCA)
rf_pack = {
    'rf': rf,
    'scaler_nut': scaler_nut,
    'scaler_suelo': scaler_suelo,
    'scaler_amb': scaler_amb,
    'pca_nut': pca_nut,
    'pca_suelo': pca_suelo,
    'pca_amb': pca_amb,
    'features': features,            # 13 características originales (entrada de la app)
    'features_pca': features_pca,    # 3 componentes (entrada real del modelo)
    'grupo_nutrientes': grupo_nutrientes,
    'grupo_suelo': grupo_suelo,
    'grupo_ambiente': grupo_ambiente
}

nombre_rf = 'modelo_random_forest_pca.joblib'
joblib.dump(rf_pack, nombre_rf)
print(f"✅ Random Forest exportado correctamente en '{nombre_rf}'.")

✅ Isolation Forest exportado correctamente en 'modelo_isolation_forest.joblib'.
✅ Random Forest exportado correctamente en 'modelo_random_forest_pca.joblib'.
